In [0]:
from pyspark.sql.functions import *

# Inspect Silver customers to confirm columns before mapping
display(spark.sql("SELECT * FROM ecommerce_dev.silver.customers LIMIT 5"))

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,bronze_ingested_at
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,Franca,SP,2026-08-04T00:54:52.697Z
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,Sao Bernardo Do Campo,SP,2026-08-04T00:54:52.697Z
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,Sao Paulo,SP,2026-08-04T00:54:52.697Z
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,Mogi Das Cruzes,SP,2026-08-04T00:54:52.697Z
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,Campinas,SP,2026-08-04T00:54:52.697Z


#### Step 1: Create the target table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ecommerce_dev.gold.dim_customer (
    customer_key BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id STRING,
    customer_unique_id STRING,
    customer_zip_code_prefix STRING,
    customer_city STRING,
    customer_state STRING,
    gold_updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Gold customer dimension - SCD Type 1 (overwrite on change, no history tracked). Grain: one row per customer_id.';

#### Step 2: MERGE from Silver (idempotent — safe to rerun)

In [0]:
from pyspark.sql.functions import *

silver_customers = spark.table('ecommerce_dev.silver.customers')
silver_customers.createOrReplaceTempView('stg_customers')

spark.sql("""
          MERGE INTO ecommerce_dev.gold.dim_customer AS target
          USING stg_customers AS source
          ON target.customer_id = source.customer_id

          WHEN MATCHED AND (
              target.customer_unique_id <> source.customer_unique_id OR
              target.customer_zip_code_prefix <> source.customer_zip_code_prefix OR
              target.customer_city <> source.customer_city OR
              target.customer_state <> source.customer_state 
          ) THEN UPDATE SET
                customer_unique_id = source.customer_unique_id,
                customer_zip_code_prefix = source.customer_zip_code_prefix,
                customer_city = source.customer_city,
                customer_state = source.customer_state,
                gold_updated_at = current_timestamp()

          WHEN NOT MATCHED THEN INSERT(
              customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state, gold_updated_at
          ) VALUES(
              source.customer_id, source.customer_unique_id, source.customer_zip_code_prefix, source.customer_city, source.customer_state, current_timestamp()
          )
          """)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

#### Row count sanity check - should match Silver customers count exactly (1:1, no fan-out)

In [0]:
silver_ct = spark.table("ecommerce_dev.silver.customers").count()
gold_ct = spark.table("ecommerce_dev.gold.dim_customer").count()
print(f"Silver: {silver_ct} | Gold: {gold_ct} | Match: {silver_ct == gold_ct}")

display(spark.sql("SELECT * FROM ecommerce_dev.gold.dim_customer LIMIT 5"))

Silver: 99441 | Gold: 99441 | Match: True


customer_key,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,gold_updated_at
1,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,Franca,SP,2026-08-09T17:09:23.341Z
2,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,Sao Bernardo Do Campo,SP,2026-08-09T17:09:23.341Z
3,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,Sao Paulo,SP,2026-08-09T17:09:23.341Z
4,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,Mogi Das Cruzes,SP,2026-08-09T17:09:23.341Z
5,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,Campinas,SP,2026-08-09T17:09:23.341Z


In [0]:
%sql
-- Set customer_key to NOT NULL before adding primary key constraint
ALTER TABLE ecommerce_dev.gold.dim_customer 
ALTER COLUMN customer_key SET NOT NULL;

ALTER TABLE ecommerce_dev.gold.dim_customer 
ADD CONSTRAINT pk_dim_customer PRIMARY KEY (customer_key);

COMMENT ON TABLE ecommerce_dev.gold.dim_customer IS 
'Gold customer dimension - SCD Type 1. Overwrites on attribute change, no history. Grain: one row per customer_id. Surrogate key: customer_key.';